In [1]:
import json

import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from tqdm.auto import tqdm
from xgboost import XGBRegressor


# === Load data ===
with open("data/nq_2024_11_07_12_14_naive/results/rag_evaluation_results_temp_940.json") as f1:
    rag_data = json.load(f1)
with open("data/nq_2024_11_07_12_14_naive/results/seper_results_temp_940.json") as f2:
    seper_data = json.load(f2)

# === Define models ===
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42),
    "ExtraTrees": ExtraTreesRegressor(n_estimators=100, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=42, tree_method='gpu_hist', predictor='gpu_predictor', verbosity=0),
    "Poly+LR": make_pipeline(PolynomialFeatures(degree=2, include_bias=False), LinearRegression()),
}

# === Clean & prepare per-metric data ===
seper_map = {item["id"]: item for item in seper_data}
all_metrics = [m for m in rag_data[0]["naive_metrics"].keys() if m != "PTrue"]
cleaned_data = {}
nan_records = []

for metric in all_metrics:
    rows = []
    for rag_item in rag_data:
        qid = rag_item["id"]
        if qid not in seper_map:
            continue
        seper_item = seper_map[qid]
        try:
            naive_value = rag_item["naive_metrics"][metric]
            rag_docs = rag_item["individual_doc_results"]
            seper_docs = seper_item["individual_doc_results"]

            for doc_idx, (rag_doc, seper_doc) in enumerate(zip(rag_docs, seper_docs)):
                doc_value = rag_doc["metrics"].get(metric)
                seper_reduction = seper_doc["utility"]["seper_reduction"]

                if not np.isfinite(naive_value) or not np.isfinite(doc_value) or not np.isfinite(seper_reduction):
                    nan_records.append({"metric": metric, "question_id": qid, "doc_index": doc_idx})
                    continue

                avg_val = (doc_value + naive_value) / 2
                diff = doc_value - naive_value
                ratio = doc_value / (naive_value + 1e-8)

                row = {
                    "question_id": qid,
                    "doc_index": doc_idx,
                    "naive": naive_value,
                    "doc": doc_value,
                    "diff": diff,
                    "ratio": ratio,
                    "avg": avg_val,
                    "diff_abs": abs(diff),
                    "naive_sq": naive_value ** 2,
                    "doc_sq": doc_value ** 2,
                    "doc_minus_avg": doc_value - avg_val,
                    "min_val": min(doc_value, naive_value),
                    "max_val": max(doc_value, naive_value),
                    "harmonic_mean": (2 * naive_value * doc_value) / (naive_value + doc_value + 1e-8),
                    "ratio_log": np.log1p(ratio),
                    "doc_normalized": doc_value / (avg_val + 1e-8),
                    "seper_reduction": seper_reduction,
                }

                # 过滤这行是否包含 nan
                if all(np.isfinite(v) for v in row.values() if isinstance(v, (int, float))):
                    rows.append(row)
                else:
                    nan_records.append({"metric": metric, "question_id": qid, "doc_index": doc_idx})

        except KeyError:
            continue

    if len(rows) >= 10:
        cleaned_data[metric] = pd.DataFrame(rows)

# === Training function ===
results = []
previews = {}

def run_regression(metric, model_name, model):
    df = cleaned_data[metric].copy()
    X = df.drop(columns=["question_id", "doc_index", "seper_reduction"])
    y = df["seper_reduction"]

    model.fit(X, y)
    y_pred = model.predict(X)

    result = {
        "metric": metric,
        "model": model_name,
        "r2_score": r2_score(y, y_pred),
        "mse": mean_squared_error(y, y_pred),
        "n_samples": len(df),
    }

    preview = df[["question_id", "doc_index"]].copy()
    preview["ground_truth"] = y
    preview["predicted"] = y_pred
    return result, preview.head(10)

# === Run all ===
for metric in tqdm(cleaned_data.keys(), desc="Metrics"):
    for model_name, model in tqdm(models.items(), leave=False, desc=f"{metric}"):
        res, preview = run_regression(metric, model_name, model)
        if res:
            results.append(res)
            previews[(metric, model_name)] = preview

# === Output summary ===
summary_df = pd.DataFrame(results).sort_values(by=["metric", "r2_score"], ascending=[True, False])
print("\n===== Summary of All Models per Metric =====")
print(summary_df.round(4))

/tmp/ipykernel_919230/2240006278.py:75: RuntimeWarning: invalid value encountered in log1p
  "ratio_log": np.log1p(ratio),


Metrics:   0%|          | 0/19 [00:00<?, ?it/s]

MaximumSequenceProbability:   0%|          | 0/8 [00:00<?, ?it/s]

/home/yijiexu/micromamba/envs/semantic_uncertainty/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.301e+01, tolerance: 5.250e-02
  model = cd_fast.enet_coordinate_descent(


Perplexity:   0%|          | 0/8 [00:00<?, ?it/s]

...

Eccentricity_NLI_score_entail:   0%|          | 0/8 [00:00<?, ?it/s]

LexicalSimilarity_rougeL:   0%|          | 0/8 [00:00<?, ?it/s]

KernelLanguageEntropy:   0%|          | 0/8 [00:00<?, ?it/s]

LUQ:   0%|          | 0/8 [00:00<?, ?it/s]


===== Summary of All Models per Metric =====
      metric             model  r2_score     mse  n_samples
68       CCP        ExtraTrees    0.1756  0.0872       4916
67       CCP      RandomForest    0.1436  0.0906       4916
70       CCP           XGBoost    0.0961  0.0956       4916
69       CCP  GradientBoosting    0.0478  0.1007       4916
71       CCP           Poly+LR    0.0045  0.1053       4916
..       ...               ...       ...     ...        ...
77  TokenSAR  GradientBoosting    0.1424  0.0910       4947
79  TokenSAR           Poly+LR    0.0075  0.1053       4947
72  TokenSAR  LinearRegression    0.0027  0.1059       4947
73  TokenSAR             Ridge    0.0025  0.1059       4947
74  TokenSAR             Lasso    0.0000  0.1061       4947

[152 rows x 5 columns]


In [2]:
summary_df

,metric,model,r2_score,mse,n_samples
68,CCP,ExtraTrees,0.175557,0.087195,4916
67,CCP,RandomForest,0.143559,0.090580,4916
70,CCP,XGBoost,0.096093,0.095600,4916
69,CCP,GradientBoosting,0.047797,0.100708,4916
71,CCP,Poly+LR,0.004507,0.105286,4916
...,...,...,...,...,...
77,TokenSAR,GradientBoosting,0.142363,0.091025,4947
79,TokenSAR,Poly+LR,0.007527,0.105336,4947
72,TokenSAR,LinearRegression,0.002671,0.105851,4947
73,TokenSAR,Ridge,0.002464,0.105873,4947


In [3]:
# === 用于在 Jupyter 中展示每个模型+metric的 top 5 预测 vs 实际值 ===

from IPython.display import display, HTML

print("🔍 Showing top-5 predictions for each (metric, model) in ranked order:\n")

for _, row in summary_df.iterrows():
    metric = row["metric"]
    model = row["model"]

    key = (metric, model)
    if key not in previews:
        continue

    print(f"📌 Metric: {metric} | Model: {model}")
    print(f"R²: {row['r2_score']:.4f} | MSE: {row['mse']:.6f} | Samples: {int(row['n_samples'])}")

    preview_df = previews[key].copy()
    preview_df.columns = ["question_id", "doc_index", "ground_truth", "predicted"]
    display(HTML(preview_df.to_html(index=False)))

🔍 Showing top-5 predictions for each (metric, model) in ranked order:

📌 Metric: CCP | Model: ExtraTrees
R²: 0.1756 | MSE: 0.087195 | Samples: 4916


question_id,doc_index,ground_truth,predicted
test_2619,0,0.990489,0.114635
test_2619,1,0.004199,0.004199
test_2619,2,0.990489,0.114635
test_2619,3,0.219955,0.114635
test_2619,4,0.002516,0.114635
test_2619,5,0.002043,0.114635
test_2619,6,0.000757,0.000757
test_2619,7,0.000882,0.101614
test_2619,8,0.002546,0.114635
test_2619,9,0.363815,0.114635


📌 Metric: CCP | Model: RandomForest
R²: 0.1436 | MSE: 0.090580 | Samples: 4916


question_id,doc_index,ground_truth,predicted
test_2619,0,0.990489,0.113986
test_2619,1,0.004199,0.020190
test_2619,2,0.990489,0.113986
test_2619,3,0.219955,0.113986
test_2619,4,0.002516,0.113986
test_2619,5,0.002043,0.113986
test_2619,6,0.000757,0.011609
test_2619,7,0.000882,0.094937
test_2619,8,0.002546,0.113986
test_2619,9,0.363815,0.113986


📌 Metric: CCP | Model: XGBoost
R²: 0.0961 | MSE: 0.095600 | Samples: 4916


...

question_id,doc_index,ground_truth,predicted
test_2619,0,0.990489,0.121627
test_2619,1,0.004199,0.099457
test_2619,2,0.990489,0.118984
test_2619,3,0.219955,0.123151
test_2619,4,0.002516,0.106828
test_2619,5,0.002043,0.104157
test_2619,6,0.000757,0.098192
test_2619,7,0.000882,0.122763
test_2619,8,0.002546,0.090486
test_2619,9,0.363815,0.123201


📌 Metric: TokenSAR | Model: Ridge
R²: 0.0025 | MSE: 0.105873 | Samples: 4947


question_id,doc_index,ground_truth,predicted
test_2619,0,0.990489,0.110923
test_2619,1,0.004199,0.089891
test_2619,2,0.990489,0.106116
test_2619,3,0.219955,0.120784
test_2619,4,0.002516,0.094525
test_2619,5,0.002043,0.092733
test_2619,6,0.000757,0.089181
test_2619,7,0.000882,0.114503
test_2619,8,0.002546,0.085237
test_2619,9,0.363815,0.120143


📌 Metric: TokenSAR | Model: Lasso
R²: 0.0000 | MSE: 0.106134 | Samples: 4947


question_id,doc_index,ground_truth,predicted
test_2619,0,0.990489,0.116675
test_2619,1,0.004199,0.116675
test_2619,2,0.990489,0.116675
test_2619,3,0.219955,0.116675
test_2619,4,0.002516,0.116675
test_2619,5,0.002043,0.116675
test_2619,6,0.000757,0.116675
test_2619,7,0.000882,0.116675
test_2619,8,0.002546,0.116675
test_2619,9,0.363815,0.116675


In [3]:
# === Sample prediction output ===
print("\n===== Sample Predictions =====")
for (metric, model_name), preview in list(previews.items())[:5]:
    print(f"\n--- {metric} | {model_name} ---")
    print(preview.to_string(index=False))


===== Sample Predictions =====

--- MaximumSequenceProbability | LinearRegression ---
question_id  doc_index  seper_reduction  predicted
  test_2619          0         0.990489   0.086926
  test_2619          1         0.004199   0.119428
  test_2619          2         0.990489   0.090348
  test_2619          3         0.219955   0.079978
  test_2619          4         0.002516   0.098800

--- MaximumSequenceProbability | Ridge ---
question_id  doc_index  seper_reduction  predicted
  test_2619          0         0.990489   0.086926
  test_2619          1         0.004199   0.119429
  test_2619          2         0.990489   0.090348
  test_2619          3         0.219955   0.079979
  test_2619          4         0.002516   0.098800

--- MaximumSequenceProbability | Lasso ---
question_id  doc_index  seper_reduction  predicted
  test_2619          0         0.990489   0.089044
  test_2619          1         0.004199   0.123088
  test_2619          2         0.990489   0.092628
  test_26